# Image denoising with DFT matrices on CIM

Use a discrete Fourier transform (DFT), a low-pass frequency mask, and an inverse DFT to denoise a real grayscale photograph. No neural network or training is involved. Compare NumPy's FFT implementation with real matrix operations mapped onto MemFast arrays, first with ideal devices and then with write variation of 0.02.

Install MemFast in this Python environment, plus `numpy`, `Pillow`, `matplotlib`, and a Jupyter frontend. The photograph ships with Matplotlib; no image download is needed. Run all cells in order. The default is CPU execution on a 64 x 64 image; set `DEVICE = 'cuda:0'` to use an available CUDA device.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.cbook import get_sample_data
from PIL import Image

from memfast import Config, Nonidealities, Simulator

DEVICE = "cpu"
SIZE = 64  # Use 32 for a smaller CPU exercise.
SEED = 135
NOISE_STD = 0.15  # Pixel values are in [0, 1].
CUTOFF = 0.18  # Radial frequency in cycles per pixel.
print(f"Device: {DEVICE}; image: {SIZE} x {SIZE}")

## 1. Load the photograph and add noise

Convert Matplotlib's Grace Hopper sample to grayscale and resize it before adding seeded Gaussian noise. Clip noisy pixels to [0, 1]. This resized photograph is the clean target for every PSNR calculation.

In [ ]:
sample_path = get_sample_data("grace_hopper.jpg", asfileobj=False)
with Image.open(sample_path) as photograph:
    grayscale = photograph.convert("L").resize((SIZE, SIZE), resample=Image.Resampling.LANCZOS)
    clean = np.asarray(grayscale, dtype=np.float64) / 255.0
rng = np.random.default_rng(SEED)
noisy = np.clip(clean + rng.normal(0.0, NOISE_STD, clean.shape), 0.0, 1.0)
assert clean.shape == noisy.shape == (SIZE, SIZE)
assert np.isfinite(clean).all() and np.isfinite(noisy).all()
print(f"Clean range: [{clean.min():.3f}, {clean.max():.3f}]")

## 2. Define the normalized DFT and the frequency mask

For an $N \times N$ image $X$, define the unitary DFT matrix

$$F_{k,n} = \frac{1}{\sqrt{N}}\exp\left(-2\pi i\frac{kn}{N}\right).$$

The two-dimensional transform and inverse are

$$Y = F X F^T, \qquad \widehat{X} = F^H (M \odot Y)\,\overline{F}.$$

Here $T$ means transpose, $H$ means conjugate transpose, and the bar means complex conjugation. Each axis uses the $1/\sqrt{N}$ normalization, matching NumPy's `norm='ortho'` in both directions.

Keep frequencies satisfying $\sqrt{f_y^2+f_x^2} \leq 0.18$ cycles/pixel. The mask uses unshifted `fftfreq` order, so the DC term is at index (0, 0); no `fftshift` belongs in the computation. A hard cutoff removes high-frequency noise but can blur detail and introduce ringing. The clean image is used only for evaluation, not to choose the cutoff.

In [ ]:
indices = np.arange(SIZE)
F = np.exp(-2j * np.pi * np.outer(indices, indices) / SIZE) / np.sqrt(SIZE)
frequencies = np.fft.fftfreq(SIZE)
radius = np.hypot(frequencies[:, None], frequencies[None, :])
mask = (radius <= CUTOFF).astype(np.float64)

numpy_spectrum = np.fft.fft2(noisy, norm="ortho")
numpy_inverse = np.fft.ifft2(mask * numpy_spectrum, norm="ortho")
standard = np.clip(numpy_inverse.real, 0.0, 1.0)

# Verify signs, normalization, both axes, and the inverse independently.
np.testing.assert_allclose(F @ noisy @ F.T, numpy_spectrum, atol=1e-12, rtol=1e-12)
np.testing.assert_allclose(F.conj().T @ numpy_spectrum @ F.conj(), noisy, atol=1e-12, rtol=1e-12)
print(f"Mask keeps {int(mask.sum())} / {mask.size} frequency bins.")
print(f"NumPy inverse imaginary RMS: {np.sqrt(np.mean(numpy_inverse.imag**2)):.3e}")

## 3. Set ideal and write-variation configurations

Both runs use signed 8-bit inputs and weights, binary slices, 64 x 64 arrays, and 8-bit ADCs. At most 64 rows contribute to each binary-slice read, within the ADC's 0..255 count range for ideal devices. Weight scales are per output channel and input scales are per input vector.

**Ideal devices still quantize.** Compare them with a dequantized matrix-product oracle, not with an assumption of exact floating-point DFT output. `Nonidealities()` currently defaults to write variation of 0.02, so the ideal case explicitly sets it to zero. The second case enables only write variation of 0.02; read variation and other nonidealities remain zero. Names and seeds make programming reproducible within each configuration.

In [ ]:
common = dict(
    activation_bits=8,
    weight_bits=8,
    input_slice=(1,) * 8,
    weight_slice=(1,) * 8,
    array_size=(64, 64),
    adc_bits=8,
    input_quant_gran=(1, None),
    weight_quant_gran=(None, 1),
    seed=SEED,
)
ideal_config = Config(**common, nonidealities=Nonidealities(write_variation=0.0))
write_config = Config(**common, nonidealities=Nonidealities(write_variation=0.02))
print("Ideal:", ideal_config.nonidealities)
print("Write 0.02:", write_config.nonidealities)

## 4. Map complex transforms to real CIM matrices

For a complex transform $B=C+iD$ and a row vector $z=a+ib$,

$$zB^T = (aC^T-bD^T) + i(aD^T+bC^T).$$

`simulator.map(weight)` accepts `[outputs, inputs]` and computes `x @ weight.T`. Map $C$ and $D$ separately, then make four real CIM calls per complex axis transform. Moving the selected image axis to the last dimension makes each row a vector along that axis. Use $B=F$ for the forward transform and $B=F^H$ for the inverse.

All forward and inverse matrix products below pass through MemFast. Complex assembly, subtraction/addition, axis movement, frequency masking, and final clipping are digital operations. FFT calls appear only in the NumPy reference.

In [ ]:
def map_complex_matrix(simulator, basis, name):
    real = simulator.map(
        torch.tensor(basis.real, dtype=torch.float32, device=simulator.device),
        name=f"{name}.real",
    )
    imag = simulator.map(
        torch.tensor(basis.imag, dtype=torch.float32, device=simulator.device),
        name=f"{name}.imag",
    )
    return real, imag


def transform_axis(values, maps, axis):
    real_map, imag_map = maps
    real = values.real.movedim(axis, -1).contiguous()
    imag = values.imag.movedim(axis, -1).contiguous()
    transformed = torch.complex(
        real_map(real) - imag_map(imag),
        imag_map(real) + real_map(imag),
    )
    return transformed.movedim(-1, axis)


@torch.no_grad()
def cim_denoise(image, frequency_mask, config, device="cpu"):
    image = np.asarray(image, dtype=np.float64)
    assert image.ndim == 2 and image.shape[0] == image.shape[1]
    assert frequency_mask.shape == image.shape
    assert np.isfinite(image).all() and np.isfinite(frequency_mask).all()
    n = image.shape[0]
    coordinates = np.arange(n)
    basis = np.exp(-2j * np.pi * np.outer(coordinates, coordinates) / n) / np.sqrt(n)
    simulator = Simulator(config, device=device)
    forward_maps = map_complex_matrix(simulator, basis, "dft.forward")
    inverse_maps = map_complex_matrix(simulator, basis.conj().T, "dft.inverse")
    pixels = torch.tensor(image, dtype=torch.float32, device=simulator.device)
    values = torch.complex(pixels, torch.zeros_like(pixels))
    spectrum = transform_axis(transform_axis(values, forward_maps, axis=1), forward_maps, axis=0)
    filtered = spectrum * torch.tensor(frequency_mask, dtype=torch.float32, device=simulator.device)
    restored = transform_axis(transform_axis(filtered, inverse_maps, axis=1), inverse_maps, axis=0)
    return restored.cpu().numpy()

## 5. Check the mapping and compare PSNR

The ideal-device check below explicitly quantizes and dequantizes signed test vectors and one real DFT matrix, then compares their digital product with a CIM read. This catches mapping/scaling errors without treating quantization as a device failure.

For pixels in [0, 1], $\mathrm{PSNR}=10\log_{10}(1/\mathrm{MSE})$, with MSE measured against the clean image. Use float64 differences and floor MSE at the smallest positive normal float64 value to keep even exact-match PSNR finite. Take the real part and clip to [0, 1] identically for all reconstructions before PSNR and display. Also report imaginary RMS before this projection: quantization and independently programmed real/imaginary maps can break conjugate symmetry.

`diagnostic(device='cpu')` is callable after these cells are executed, including when code cells are extracted into a Python namespace. It returns reconstructed images and metrics and checks shapes and finite values. It does not assert that filtering or write variation improves PSNR.

In [ ]:
def psnr(target, estimate):
    target = np.asarray(target, dtype=np.float64)
    estimate = np.asarray(estimate, dtype=np.float64)
    assert target.shape == estimate.shape
    assert np.isfinite(target).all() and np.isfinite(estimate).all()
    mse = np.mean(np.square(target - estimate))
    return float(-10.0 * np.log10(max(float(mse), np.finfo(np.float64).tiny)))


def diagnostic(device="cpu"):
    simulator = Simulator(ideal_config, device=device)
    weight = torch.tensor(F.real, dtype=torch.float32, device=simulator.device)
    vectors = torch.linspace(-1, 1, 3 * SIZE, device=simulator.device).reshape(3, SIZE)
    weight_scale = weight.abs().amax(dim=1, keepdim=True) / 127
    input_scale = vectors.abs().amax(dim=1, keepdim=True) / 127
    dequant_weight = (weight / weight_scale).round().clamp(-127, 127) * weight_scale
    dequant_input = (vectors / input_scale).round().clamp(-127, 127) * input_scale
    expected = dequant_input @ dequant_weight.T
    actual = simulator.map(weight, name="dft.ideal_check")(vectors)
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=2e-5)
    print(f"Ideal dequantized product max error: {(actual - expected).abs().max().item():.3e}")

    results = {
        "Noisy": {"image": noisy, "psnr_db": psnr(clean, noisy)},
        "NumPy low-pass": {"image": standard, "psnr_db": psnr(clean, standard)},
    }
    for label, config in [("CIM ideal", ideal_config), ("CIM write 0.02", write_config)]:
        raw = cim_denoise(noisy, mask, config, device=device)
        assert raw.shape == clean.shape and np.isfinite(raw).all()
        image = np.clip(raw.real, 0.0, 1.0)
        results[label] = dict(
            image=image,
            psnr_db=psnr(clean, image),
            imaginary_rms=float(np.sqrt(np.mean(raw.imag.astype(np.float64) ** 2))),
            rmse_vs_numpy=float(np.sqrt(np.mean((image - standard) ** 2))),
        )
    assert all(np.isfinite(item["psnr_db"]) for item in results.values())
    return results


results = diagnostic(device=DEVICE)
print(f"\n{'Method':<20} {'PSNR (dB)':>12} {'RMSE vs NumPy':>16} {'Imaginary RMS':>16}")
for label, item in results.items():
    rmse = f"{item['rmse_vs_numpy']:.6f}" if "rmse_vs_numpy" in item else "-"
    imaginary = f"{item['imaginary_rms']:.3e}" if "imaginary_rms" in item else "-"
    print(f"{label:<20} {item['psnr_db']:>12.3f} {rmse:>16} {imaginary:>16}")
write_delta = np.max(np.abs(results["CIM write 0.02"]["image"] - results["CIM ideal"]["image"]))
print(f"Write-0.02 vs ideal max pixel difference: {write_delta:.3e}")

## 6. Inspect the reconstructions

All panels use the same grayscale range. Compare background grain, edges, and fine detail. The NumPy-to-ideal difference includes finite input/weight precision at every mapped stage. The write-0.02 result additionally includes programming variation in both forward and inverse matrices. Small current perturbations can round to the same ADC counts, so the ideal and write-0.02 images may coincide; the printed pixel difference makes this visible. PSNR describes this image, mask, and seed; a different image or cutoff can change whether filtering helps.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(clean, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
axes[0].set_title("Clean target")
axes[0].axis("off")
for ax, (label, item) in zip(axes[1:], results.items()):
    ax.imshow(item["image"], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"{label}\nPSNR {item['psnr_db']:.2f} dB")
    ax.axis("off")
plt.show()